In [ ]:
from datetime import datetime
from getpass import getpass
import random

rdm_url = 'staging.rdm.example.com'
idp_name_1 = None  # 'GakuNin RDM IdP'
idp_username_1 = None
idp_password_1 = None
rdm_project_name = 'TEST-WikiDecoration-{}'.format(datetime.now().strftime('%Y%m%d'))
target_storage_name = 'NII Storage'
target_storage_id = 'osfstorage'
delete_project = True
default_result_path = None
close_on_fail = False
transition_timeout = 60000
new_wikiname = 'New Wiki'

In [ ]:
if idp_username_1 is None:
    idp_username_1 = input(prompt=f'Username for {idp_name_1}')
if idp_password_1 is None:
    idp_password_1 = getpass(prompt=f'Password for {idp_username_1}@{idp_name_1}')
(len(idp_username_1), len(idp_password_1))

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

# WikiDecoration

- サブシステム名: アドオン
- ページ/アドオン: WikiDecoration
- 機能分類: Wiki操作
- シナリオ名: WikiDecoration
- 用意するテストデータ: URL一覧、アカウント(既存ユーザー1: GRDM)

In [ ]:
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)

## ウェブブラウザの新規プライベートウィンドウでGRDMトップページを表示する

GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url)

    # 同意する ボタンが現れるまで待つ
    await expect(page.locator('//button[text() = "同意する"]')).to_be_visible(timeout=transition_timeout)

    # 同意する をクリック
    await page.locator('//button[text() = "同意する"]').click()

    # 同意する が表示されなくなったことを確認
    await expect(page.locator('//button[text() = "同意する"]')).to_have_count(0, timeout=500)

await run_pw(_step)

## ログイン情報を用いてGakuNin RDMにログインする

(IdPに関するログイン情報が与えられた場合、)
GakuNin Embeded DSのプルダウンを展開し、IdPリストから指定されたIdPを選択する。その後、アカウントのID/Passwordを入力して「Login」ボタンを押下する。

(IdPが指定されていない場合、)
CASのログイン操作を実施する。

In [ ]:
async def _step(page):
    await scripts.grdm.login(
        page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout
    )

    await scripts.grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)

## プロジェクト一覧に指定されたタイトルのプロジェクトがない場合、指定された名前のプロジェクトを作成する

プロジェクト一覧に当該プロジェクト名が表示されていない場合、「新規プロジェクト作成」をクリックし、その名前を入力、「作成」をクリックする。

In [ ]:
async def _step(page):
    await expect(page.locator('//*[@data-test-create-project-modal-button]')).to_have_count(1)

    await scripts.grdm.ensure_project_exists(page, rdm_project_name, transition_timeout=transition_timeout)

await run_pw(_step)

## 対象のプロジェクトをクリックする

プロジェクトが表示されること

In [ ]:
async def _step(page):
    await page.locator(f'//*[@data-test-dashboard-item-title and text()="{rdm_project_name}"]').click()        

    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

    await page.locator('//h3[text()="最近の活動"]').click()

await run_pw(_step)

## Wikiタブをクリックする

画面がWikiに切り替わること

In [ ]:
async def _step(page):
    await page.get_by_role("link", name="Wiki", exact=True).click()
    await expect(page.locator('//*[contains(@class, "title-text")]//*[text() = "プロジェクトのWiki"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 画面左のプロジェクトのWikiの上にある「新規」ボタンをクリックする

新規Wikiページを追加画面が表示される

In [ ]:
async def _step(page):
    await page.locator('div[data-target="#newWiki"]').click() 
    await expect(page.locator('h3.modal-title', has_text="新規Wikiページを追加")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 同階層に追加を選択し、任意の新規Wiki名を入力して「追加」ボタンをクリックする

画面左のプロジェクトのWikiに、追加したWikiが表示される

In [ ]:
async def _step(page):
    await page.locator('input[name="addHierarchy"][value="same"]').check()
    await page.fill('#data', new_wikiname)
    await page.locator('#add-wiki-submit').click()
    await expect(page.locator(f'//*[contains(@class, "title-text")]//*[text() = "{new_wikiname}"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「プロジェクトのWiki」内にある追加したWikiを選択

Wikiの内容が表示されること（初期表示は空白で「重要な情報、リンク、または画像をここに追加して、プロジェクトを説明します。」の記載があること）

In [ ]:
async def _step(page):
    await grdm.open_wiki(page, new_wikiname, '重要な情報、リンク、または画像をここに追加して、プロジェクトを説明します。')

await run_pw(_step)

## 「編集」ボタンをクリック

Wikiの編集画面が表示されること

In [ ]:
async def _step(page):
    await grdm.open_edit_wiki(page)

await run_pw(_step)

## 編集画面にて「test」と入力後、編集画面の戻るボタンをクリックする。	

編集した結果が1つ前に戻り入力した「test」が削除されること

In [ ]:
text = 'test'

async def _step(page):
    await grdm.fill_text(page, text)
    await page.locator('#undoWiki').click()
    await expect(page.locator('#mEditor .ProseMirror[contenteditable="true"]')).to_have_text("", timeout=transition_timeout)
    await expect(page.locator('#redoWiki')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## 編集画面の進むボタンをクリックする

編集した結果が1つ前に進み削除された「test」が表示されること

In [ ]:
async def _step(page):
    await page.locator('#redoWiki').click()
    editor_locator = page.locator('#mEditor .ProseMirror[contenteditable="true"]')
    await expect(page.locator('#mEditor .ProseMirror[contenteditable="true"]')).to_have_text(text, timeout=transition_timeout)

await run_pw(_step)

## 入力した「test」の文字を範囲指定して編集画面の太字ボタンをクリックし保存ボタンをクリック

入力した「test」が太文字表示になること

In [ ]:
async def _step(page):
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['strongWiki'])

    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator("strong", has_text=text)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「編集」ボタンをクリック後、入力した「test」の文字を範囲指定して編集画面の太字ボタンをクリックし保存ボタンをクリック

入力した「test」が太字表示ではなくなること

In [ ]:
async def _step(page):
    await grdm.open_edit_wiki(page)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['strongWiki'])

    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator("p", has_text=text)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「編集」ボタンをクリック後、入力した「test」の文字を範囲指定して編集画面の斜体ボタンをクリックし保存ボタンをクリック

入力した「test」が斜体表示になること

In [ ]:
async def _step(page):
    await grdm.open_edit_wiki(page)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['italicWiki'])

    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator("em", has_text=text)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「編集」ボタンをクリック後、入力した「test」の文字を範囲指定して編集画面の斜体ボタンをクリックし保存ボタンをクリック

入力した「test」が斜体表示ではなくなること

In [ ]:
async def _step(page):
    await grdm.open_edit_wiki(page)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['italicWiki'])

    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator("p", has_text=text)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「編集」ボタンをクリック後、入力した「test」の文字を範囲指定して編集画面の取り消し線文字ボタンをクリックし保存ボタンをクリック

入力した「test」に取消線がついて表示されること

In [ ]:
async def _step(page):
    await grdm.open_edit_wiki(page)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['format_strikethrough'])

    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator("del", has_text=text)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「編集」ボタンをクリック後、入力した「test」の文字を範囲指定して編集画面の取り消し線文字ボタンをクリックし保存ボタンをクリック

入力した「test」に取消線がなくなって表示されること

In [ ]:
async def _step(page):
    await grdm.open_edit_wiki(page)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['format_strikethrough'])

    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator("p", has_text=text)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「編集」ボタンをクリック後、入力した「test」の文字を範囲指定して編集画面の下線文字ボタンをクリックし保存ボタンをクリック

入力した「test」に下線がついて表示されること

In [ ]:
async def _step(page):
    await grdm.open_edit_wiki(page)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['format_underlined'])

    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator("u", has_text=text)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「編集」ボタンをクリック後、入力した「test」の文字を範囲指定して編集画面の下線文字ボタンをクリックし保存ボタンをクリック

入力した「test」に下線がなくなって表示されること

In [ ]:
async def _step(page):
    await grdm.open_edit_wiki(page)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['format_underlined'])

    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator("p", has_text=text)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 入力した「test」の文字を範囲指定して編集画面のフォントカラーボタンをクリック

フォントカラーのRGB設定のウィンドウが表示されること

In [ ]:
async def _step(page):
    await grdm.open_edit_wiki(page)
    await grdm.select_text_range(page, text)

await run_pw(_step)

## 「編集」ボタンをクリック後、表示されているRGB設定画面のRの項目を255にしEnterキーを押下後、保存ボタンをクリック

入力した「test」のフォントカラーが赤で表示されること

In [ ]:
async def _step(page):
    await grdm.click_wiki_menu_save(page, ['format_color_text'])

    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator("span", has_text=text)).to_have_css("color", "rgb(255, 0, 0)")

await run_pw(_step)

## ファイルタブをクリックする

画面がファイルに切り替わること

In [ ]:
async def _step(page):
    await page.locator('#projectNavFiles a').click()
    time.sleep(1)
    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

    await expect(page.locator('//h3[text()="最近の活動"]')).not_to_be_visible()

await run_pw(_step)

## 「NII Storage」を選択し、画面上部の「フォルダのアップロード」ボタンをクリックする

フォルダ選択画面が表示される

In [ ]:
async def _step(page):
    await grdm.get_select_storage_title_locator(page, target_storage_name).click()
    await expect(page.locator('//i[contains(@class, "fa-plus")]/../*[text() = "フォルダのアップロード"]')).to_be_enabled()

await run_pw(_step)

## 事前に用意しておいた「WikiImport_Heading」フォルダを選択し「アップロード」ボタンをクリックする

ファイルのアップロード確認メッセージが表示される
> **※本手順の確認内容は自動検証できないため、次手順の中で結果を一括確認します。**

In [ ]:
# ・(補足)自動試験の場合、「確認内容」が確認できないため、手動で作業が必要
async def _step(page):
    pass

await run_pw(_step)

## 「ステータスをアップロード」画面下の「Done」ボタンをクリックする

「WikiImport_Heading」フォルダをアップロードしました

In [ ]:
foldername = 'WikiImport_Heading'
folderpath = os.path.join('resources/Datatest-Wiki', foldername)

async def _step(page):
    await grdm.upload_folder(page, folderpath)
    await asyncio.sleep(1)
    await expect(page.locator(f'//*[@role = "progressbar"]')).to_have_count(0, timeout=transition_timeout)
    await expect(grdm.get_select_folder_title_locator(page, foldername)).to_have_count(2, timeout=transition_timeout)
await run_pw(_step)

## Wikiタブをクリックする

画面がWikiに切り替わること

In [ ]:
async def _step(page):
    await page.get_by_role("link", name="Wiki", exact=True).click()
    await expect(page.locator('//*[contains(@class, "title-text")]//*[text() = "プロジェクトのWiki"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「インポート」ボタンをクリックする

「Wikiページをインポートする」画面が表示されること

In [ ]:
async def _step(page):
    await page.locator('div[data-target="#wikiImport"]').click()
    await expect(page.locator('h3.modal-title', has_text="Wikiページをインポートする")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## リストボックスで「WikiImport_Heading」を選択し「インポート」ボタンをクリックする

- ボタンが「Wikiページを検証中」→「インポート中」となること
- インポートが完了すると「Import Complete」と表示されること

In [ ]:
import_wikiname = 'WikiImport_Heading'
async def _step(page):
    await page.select_option('#wikiImportDir', label=f"{import_wikiname}")
    await grdm.click_and_expect_alert(page, lambda: page.locator('//button[text()="インポート"]').click(), "Import Complete", transition_timeout=transition_timeout)

await run_pw(_step)

## 「Import Complete」と表示された画面にて「OK」ボタンをクリックする

Wikiの画面に戻り、インポートされたファイル（WikiImport_Heading）が、画面左の「プロジェクトのWiki」配下に表示されること

In [ ]:
async def _step(page):
    await expect(page.locator(f'//*[contains(@class, "title-text")]//*[text() = "{import_wikiname}"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 画面左の「プロジェクトのWiki」内にある「WikiImport_Heading」をクリックする

画面右のWiki表示領域にWikiの内容が表示されること

In [ ]:
async def _step(page):
    await grdm.open_wiki(page, import_wikiname, '見出し１')

await run_pw(_step)

## 「WikiImport_Heading」の「編集」ボタンをクリック後、1行目を選択し、編集画面の目次ボタンをクリックして保存ボタンをクリック

見出しに対して目次が自動生成され、目次が機能すること

In [ ]:
async def _step(page):
    await grdm.open_edit_wiki(page)
    await grdm.click_wiki_menu_save(page, ['sort'])

    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)

    toc_locator = view_locator.locator('ul[data-spread="false"]')
    await expect(toc_locator).to_be_visible(timeout=transition_timeout)

    headings = ["見出し１", "見出し２", "見出し３"]
    for heading in headings:
        link = toc_locator.locator(f'a[href="#{heading}"]')
        await expect(link).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 任意のWikiを開き、「編集」ボタンをクリック後、「test」と入力し、文字を範囲指定して編集画面のリンクボタンをクリック

「リンクを追加」ウィンドウが表示されること

In [ ]:
text = 'test クリック'
async def _step(page):
    await grdm.open_wiki(page, new_wikiname, 'test')
    await grdm.open_edit_wiki(page)
    await grdm.fill_text(page, text)
    await grdm.select_text_range(page, text)
    
    await page.locator(f'#mMenuBar span:has-text("link")').click()
    await expect(page.locator('//h4[contains(@class, "modal-title") and text()="リンクを追加"]')).to_be_visible(timeout=transition_timeout*2)

await run_pw(_step)

## 「リンクを追加」ウィンドウのリンクURL項目に「../HOME」を記載、リンクツールチップ項目に「テストリンク」を記載して追加をクリック

「リンクを追加」ウィンドウが閉じられること

In [ ]:
async def _step(page):
    modal_locator = page.locator('.modal-content:has-text("リンクを追加")')
    await modal_locator.locator('#linkSrc').fill('../HOME')
    await modal_locator.locator('#linkTitle').fill('テストリンク')
    await modal_locator.locator('#addLink').click()

    await expect(modal_locator).not_to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「保存」ボタンをクリックして保存を行い、追加したリンクにカーソルを合わせる

ツールチップに「テストリンク」と表示されること

In [ ]:
async def _step(page):
    await page.locator('//input[@type="submit" and @value="保存"]').click()
    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)

    link_locator = view_locator.locator('a[href="../HOME"]')
    await expect(link_locator).to_be_visible(timeout=transition_timeout)
    await expect(link_locator).to_have_attribute("title", "テストリンク", timeout=transition_timeout)

await run_pw(_step)

## 追加したリンクにカーソルをクリック

「Wiki」のホームへリンクすること

In [ ]:
import re

async def _step(page):
    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await view_locator.locator('a[href="../HOME"]').click()
    await expect(page).to_have_url(re.compile(r"/wiki/home/"), timeout=transition_timeout)
    await expect(page.locator('//*[contains(@class, "title-text")]//*[text() = "プロジェクトのWiki"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 任意のWikiを開き、「編集」ボタンをクリック後、「test」と入力し、入力した「test」の文字を範囲指定して編集画面の引用ボタンをクリック

入力した「test」が引用（縦棒）された状態で表示されること

In [ ]:
text = 'test 引用'
async def _step(page):
    await grdm.open_wiki(page, new_wikiname, 'test')
    await grdm.open_edit_wiki(page)
    await grdm.fill_text(page, text)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['format_quote'])
    
    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator("blockquote", has_text=text)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「編集」ボタンをクリック後、「test」と入力し、入力した「test」の文字を範囲指定して編集画面のコードブロックボタンをクリック

入力した「test」がコードブロック（四角の枠線）された状態で表示されること

In [ ]:
text = 'test コードブロック'
async def _step(page):
    await grdm.open_wiki(page, new_wikiname, 'test')
    await grdm.open_edit_wiki(page)
    count = await page.locator('blockquote').count()
    if count > 0:
        await page.locator('blockquote').evaluate_all("elements => elements.forEach(el => el.remove())")
    await grdm.fill_text(page, text)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['code'])
    
    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator("code", has_text=text)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「編集」ボタンをクリック後、画像挿入ボタンをクリック

「画像を追加」ウィンドウが表示されること

In [ ]:
async def _step(page):
    await grdm.open_edit_wiki(page)
    editor_locator = page.locator('#mEditor .ProseMirror[contenteditable="true"]')
    await editor_locator.click()
    count = await page.locator('pre:has(code)').count()
    if count > 0:
        await page.locator('pre:has(code)').evaluate_all("elements => elements.forEach(el => el.remove())")
    await page.locator(f'#mMenuBar span:has-text("image")').click()
    await expect(page.locator('//h4[contains(@class, "modal-title") and text()="画像を追加"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「画像を追加」ウィンドウの項目にそれぞれ以下を入力して追加をクリック

- 画像URL：存在する画像のURL（例：https://staging2.rdm.example.com/static/img/rdm_logo-1.png）
- 画像ツールチップ：画像挿入テスト
- 代替テキスト：非表示時文言
- 画像サイズ：50
- 「画像を追加」ウィンドウが閉じられること

In [ ]:
async def _step(page):
    modal_locator = page.locator('.modal-content:has-text("画像を追加")')
    await modal_locator.locator('#imageSrc').fill('http://192.168.168.167:5000/static/img/rdm_logo-1.png')
    await modal_locator.locator('#imageTitle').fill('画像挿入テスト')
    await modal_locator.locator('#imageAlt').fill('非表示時文言')
    await modal_locator.locator('#imageWidth').fill('50')
    await modal_locator.locator('#addImage').click()

    await expect(modal_locator).not_to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「保存」ボタンをクリックして保存を行い、追加した画像にカーソルを合わせる

画像が表示され、ツールチップに「画像挿入テスト」と表示されること

In [ ]:
async def _step(page):
    await page.locator('//input[@type="submit" and @value="保存"]').click()
    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)

    img_locator = page.locator('img[alt="非表示時文言"]')
    await expect(img_locator).to_be_visible(timeout=transition_timeout)
    await expect(img_locator).to_have_attribute("title", "画像挿入テスト", timeout=transition_timeout)

await run_pw(_step)

## 「編集」ボタンをクリック後、先ほど作成した画像を削除し、再度、画像挿入ボタンをクリック

「画像を追加」ウィンドウが表示されること

In [ ]:
async def _step(page):
    await grdm.open_edit_wiki(page)
    count = await page.locator('img[alt="非表示時文言"]').count()
    if count > 0:
        await page.locator('img[alt="非表示時文言"]').evaluate_all("elements => elements.forEach(el => el.remove())")
    await page.locator(f'#mMenuBar span:has-text("image")').click()
    await expect(page.locator('//h4[contains(@class, "modal-title") and text()="画像を追加"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「画像を追加」ウィンドウの項目の値を以下に修正して追加をクリック
- 画像URL：存在する画像のURL（例：https://staging2.rdm.example.com/static/img/rdm_logo-1_none.png）
- 画像ツールチップ：画像挿入テスト
- 代替テキスト：非表示時文言
- 画像サイズ：50
- 「画像を追加」ウィンドウが閉じられること

In [ ]:
async def _step(page):
    modal_locator = page.locator('.modal-content:has-text("画像を追加")')
    await modal_locator.locator('#imageSrc').fill('http://192.168.168.167:5000/static/img/rdm_logo-1_none.png')
    await modal_locator.locator('#imageTitle').fill('画像挿入テスト')
    await modal_locator.locator('#imageAlt').fill('非表示時文言')
    await modal_locator.locator('#imageWidth').fill('50')
    await modal_locator.locator('#addImage').click()

    await expect(modal_locator).not_to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「保存」ボタンをクリックして保存を行い、追加した画像にカーソルを合わせる

画像が表示されず代わりに「非表示時文言」の表示がされ、ツールチップに「画像挿入テスト」と表示されること

In [ ]:
async def _step(page):
    await page.locator('//input[@type="submit" and @value="保存"]').click()
    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//span[contains(@class, "title-text")]//b[contains(text(), "プロジェクトのWiki")]')).to_be_visible(timeout=transition_timeout)

    img_locator = page.locator('img[alt="非表示時文言"]')
    await expect(img_locator).to_be_visible(timeout=transition_timeout)
    await expect(img_locator).to_have_attribute("title", "画像挿入テスト", timeout=transition_timeout)

await run_pw(_step)

## 「編集」ボタンをクリック後、「番号付きリスト」と入力し、入力した値を範囲指定して編集画面の番号付きリストボタンをクリックし保存ボタンをクリック

入力した「番号付きリスト」が番号付きリストとして文頭に数字がついて表示されること

In [ ]:
text = '番号付きリスト'

async def _step(page):
    await grdm.open_edit_wiki(page)
    count = await page.locator('img[alt="非表示時文言"]').count()
    if count > 0:
        await page.locator('img[alt="非表示時文言"]').evaluate_all("elements => elements.forEach(el => el.remove())")
    await grdm.fill_text(page, text)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['format_list_numbered'])
    
    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator('ol li p',has_text=f"{text}")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「編集」ボタンをクリック後、「箇条書きリスト」と入力し、入力した値を範囲指定して編集画面の箇条書きリストボタンをクリックし保存ボタンをクリック

入力した「箇条書きリスト」が箇条書きリストとして文頭に「・」がついて表示されること

In [ ]:
text = '箇条書きリスト'

async def _step(page):
    await grdm.open_edit_wiki(page)
    count = await page.locator('ol li p').count()
    if count > 0:
        await page.locator('ol li p').evaluate_all("elements => elements.forEach(el => el.remove())")
    await grdm.fill_text(page, text)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['format_list_bulleted'])
    
    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator('ul li p',has_text=f"{text}")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「編集」ボタンをクリック後、入力した「test」の文字を範囲指定して編集画面の見出しボタンをクリック

入力した「test」が見出し化された状態（大きな文字）で表示されること

In [ ]:
text = 'test 見出し'

async def _step(page):
    await grdm.open_edit_wiki(page)
    count = await page.locator('ul li p').count()
    if count > 0:
        await page.locator('ul li p').evaluate_all("elements => elements.forEach(el => el.remove())")
    await grdm.fill_text(page, text)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['view_headline'])
    
    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator(f'h1:has-text("{text}")')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「編集」ボタンをクリック後、空行を選択して水平線ボタンをクリックし保存ボタンをクリック

選択した空行が水平線にかわって表示されること

In [ ]:
async def _step(page):
    await grdm.open_edit_wiki(page)
    await grdm.fill_text(page, text)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['horizontal_rule'])
    
    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator('hr')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「編集」ボタンをクリック後、テーブル挿入ボタンをクリックし保存ボタンをクリック

テーブルが表示されること

In [ ]:
async def _step(page):
    await grdm.open_edit_wiki(page)
    count = await page.locator('hr').count()
    if count > 0:
        await page.locator('hr').evaluate_all("elements => elements.forEach(el => el.remove())")
    await grdm.click_wiki_menu_save(page, ['table'])

    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator('table')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「編集」ボタンをクリック後、テーブルを選択してテーブル挿入ボタンの横プルダウンをクリックして前に列を挿入をクリックし保存ボタンをクリック

テーブルに列が挿入されていること

In [ ]:
async def _step(page):
    table_locator = page.locator("#mView .tableWrapper table")
    col_locator =  table_locator.locator("tr").first.locator("th, td")
    before_col_number = await col_locator.count()
    print(before_col_number)

    await grdm.open_edit_wiki(page)
    row_index = 1
    col_index = 0
    await grdm.click_table_menu_save(page, row_index, col_index, "前に列を挿入")

    await expect(table_locator).to_be_visible(timeout=transition_timeout)
    after_col_number = await col_locator.count()
    print(after_col_number)
    assert after_col_number == before_col_number+1

await run_pw(_step)

## 「編集」ボタンをクリック後、テーブルを選択してテーブル挿入ボタンの横プルダウンをクリックして後に列を挿入をクリックし保存ボタンをクリック

テーブルに列が挿入されていること

In [ ]:
async def _step(page):
    table_locator = page.locator("#mView .tableWrapper table")
    col_locator =  table_locator.locator("tr").first.locator("th, td")
    before_col_number = await col_locator.count()
    print(before_col_number)

    await grdm.open_edit_wiki(page)
    row_index = 1
    col_index = 1
    await grdm.click_table_menu_save(page, row_index, col_index, "後に列を挿入")

    await expect(table_locator).to_be_visible(timeout=transition_timeout)
    after_col_number = await col_locator.count()
    print(after_col_number)
    assert after_col_number == before_col_number+1

await run_pw(_step)

## 「編集」ボタンをクリック後、テーブルを選択してテーブル挿入ボタンの横プルダウンをクリックして前に行を挿入をクリックし保存ボタンをクリック

テーブルに行が挿入されていること


In [ ]:
async def _step(page):
    table_locator = page.locator("#mView .tableWrapper table")
    row_locator =  table_locator.locator("tr")
    before_row_number = await row_locator.count()
    print(before_row_number)

    await grdm.open_edit_wiki(page)
    row_index = 1
    col_index = 1
    await grdm.click_table_menu_save(page, row_index, col_index, "前に行を挿入")

    await expect(table_locator).to_be_visible(timeout=transition_timeout)
    after_row_number = await row_locator.count()
    print(after_row_number)
    assert after_row_number == before_row_number+1

await run_pw(_step)

## 「編集」ボタンをクリック後、テーブルを選択してテーブル挿入ボタンの横プルダウンをクリックして後に行を挿入をクリックし保存ボタンをクリック

テーブルに行が挿入されていること

In [ ]:
async def _step(page):
    table_locator = page.locator("#mView .tableWrapper table")
    row_locator =  table_locator.locator("tr")
    before_row_number = await row_locator.count()
    print(before_row_number)

    await grdm.open_edit_wiki(page)
    row_index = 2
    col_index = 3
    await grdm.click_table_menu_save(page, row_index, col_index, "後に行を挿入")

    await expect(table_locator).to_be_visible(timeout=transition_timeout)
    after_row_number = await row_locator.count()
    print(after_row_number)
    assert after_row_number == before_row_number+1

await run_pw(_step)

## 「編集」ボタンをクリック後、テーブルにて縦一列を全て選択してテーブル挿入ボタンの横プルダウンをクリックしてセルを削除をクリックし保存ボタンをクリック

テーブルのセルが削除されていること

In [ ]:
async def _step(page):
    table_locator = page.locator("#mView .tableWrapper table")
    row_locator =  table_locator.locator("tr")
    before_row_number = await row_locator.count()
    print(before_row_number)

    await grdm.open_edit_wiki(page)
    row_index = 1
    col_index = 4
    await grdm.click_table_menu_save(page, row_index, col_index, "セルを削除")

    await expect(table_locator).to_be_visible(timeout=transition_timeout)
    after_row_number = await row_locator.count()
    print(after_row_number)
    assert after_row_number == before_row_number-1

await run_pw(_step)

## 「編集」ボタンをクリック後、テーブルを選択してテーブル挿入ボタンの横プルダウンをクリックしてテーブルを削除をクリックし保存ボタンをクリック

テーブルが削除されていること

In [ ]:
async def _step(page):
    await grdm.open_edit_wiki(page)
    row_index = 0
    col_index = 1
    await grdm.click_table_menu_save(page, row_index, col_index, "テーブルを削除")
    await expect(page.locator("#mView .tableWrapper table")).not_to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「編集」ボタンをクリック後、ヘルプボタンをクリック

「wiki構文のヘルプ」ウィンドウが表示されること

In [ ]:
async def _step(page):
    await grdm.open_edit_wiki(page)
    await page.locator(f'#mMenuBar span:has-text("help")').click()
    await expect(page.locator('//h3[contains(@class, "modal-title") and text()="Wiki構文のヘルプ"]')).to_be_visible(timeout=transition_timeout*2)


await run_pw(_step)

## 入力した「test」の文字を範囲指定して編集画面の太字ボタンと斜体ボタンをそれぞれクリックし保存ボタンをクリック

入力した「test」が太文字、斜体表示になること

In [ ]:
text = 'test 太文字、斜体'
async def _step(page):
    await page.locator("#wiki-help-modal button", has_text="閉じる").click()
    await grdm.fill_text(page, text)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['strongWiki', 'italicWiki'])
    
    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator("em > strong", has_text=text)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 新しく入力した「test」の文字を範囲指定して編集画面の太字ボタンと取消線ボタンをそれぞれクリックし保存ボタンをクリック

入力した「test」が太文字、取消線表示になること

In [ ]:
text = 'test 太文字、取消線'
async def _step(page):
    await grdm.open_edit_wiki(page)
    await grdm.fill_text(page, text)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['strongWiki', 'format_strikethrough'])
    
    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator("strong > del", has_text=text)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 新しく入力した「test」の文字を範囲指定して編集画面の太字ボタンと下線ボタンをそれぞれクリックし保存ボタンをクリック

入力した「test」が太文字、下線表示になること

In [ ]:
text = 'test 太文字、下線'
async def _step(page):
    await grdm.open_edit_wiki(page)
    await grdm.fill_text(page, text)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['strongWiki', 'format_underlined'])
    
    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator("strong > u", has_text=text)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 新しく入力した「test」の文字を範囲指定して編集画面の太字ボタンとフォントカラーボタンをそれぞれクリックし表示されているRGB設定画面のRの項目を255にして保存ボタンをクリック

入力した「test」が太文字、フォントカラーが赤で表示になること

In [ ]:
text = 'test 太文字、フォントカラーが赤'
async def _step(page):
    await grdm.open_edit_wiki(page)
    await grdm.fill_text(page, text)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['strongWiki', 'format_color_text'])
    
    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator("strong > span", has_text=text)).to_have_css("color", "rgb(255, 0, 0)")

await run_pw(_step)

## 新しく入力した「test」の文字を範囲指定して編集画面の斜体ボタンと取消線ボタンをそれぞれクリックし保存ボタンをクリック

入力した「test」が斜体、取消線表示になること

In [ ]:
text = 'test 斜体、取消線'
async def _step(page):
    await grdm.open_edit_wiki(page)
    await grdm.fill_text(page, text)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['italicWiki', 'format_strikethrough'])
    
    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator("em > del", has_text=text)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 新しく入力した「test」の文字を範囲指定して編集画面の斜体ボタンと下線ボタンをそれぞれクリックし保存ボタンをクリック

入力した「test」が斜体、下線表示になること

In [ ]:
text = 'test 斜体、下線'
async def _step(page):
    await grdm.open_edit_wiki(page)
    await grdm.fill_text(page, text)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['italicWiki', 'format_underlined'])
    
    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator("em > u", has_text=text)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 新しく入力した「test」の文字を範囲指定して編集画面の斜体ボタンとフォントカラーボタンをそれぞれクリックし表示されているRGB設定画面のRの項目を255にして保存ボタンをクリック

入力した「test」が斜体、フォントカラーが赤で表示になること

In [ ]:
text = 'test 斜体、フォントカラーが赤'
async def _step(page):
    await grdm.open_edit_wiki(page)
    await grdm.fill_text(page, text)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['italicWiki', 'format_color_text'])
    
    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator("em > span", has_text=text)).to_have_css("color", "rgb(255, 0, 0)")

await run_pw(_step)

## 新しく入力した「test」の文字を範囲指定して編集画面の取消線ボタンと下線ボタンをそれぞれクリックし保存ボタンをクリック

入力した「test」が取消、下線表示になること

In [ ]:
text = 'test 取消、下線'
async def _step(page):
    await grdm.open_edit_wiki(page)
    await grdm.fill_text(page, text)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['format_strikethrough', 'format_underlined'])
    
    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator("u > del", has_text=text)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 新しく入力した「test」の文字を範囲指定して編集画面の取消ボタンとフォントカラーボタンをそれぞれクリックし表示されているRGB設定画面のRの項目を255にして保存ボタンをクリック

入力した「test」が取消、フォントカラーが赤で表示になること

In [ ]:
text = 'test 取消、フォントカラーが赤'
async def _step(page):
    await grdm.open_edit_wiki(page)
    await grdm.fill_text(page, text)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['format_strikethrough', 'format_color_text'])
    
    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator("span > del", has_text=text)).to_have_css("color", "rgb(255, 0, 0)")

await run_pw(_step)

## 新しく入力した「test」の文字を範囲指定して編集画面の下線ボタンとフォントカラーボタンをそれぞれクリックし表示されているRGB設定画面のRの項目を255にして保存ボタンをクリック

入力した「test」が下線、フォントカラーが赤で表示になること

In [ ]:
text = 'test 下線、フォントカラーが赤'
async def _step(page):
    await grdm.open_edit_wiki(page)
    await grdm.fill_text(page, text)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['format_underlined', 'format_color_text'])
    
    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator("u > span", has_text=text)).to_have_css("color", "rgb(255, 0, 0)")

await run_pw(_step)

## 新しく入力した「test」の文字を範囲指定して編集画面の太字ボタンと斜体ボタンと取消線ボタンをそれぞれクリックし保存ボタンをクリック

入力した「test」が太文字、斜体、取消線表示になること

In [ ]:
text = 'test 太文字、斜体、取消線'
async def _step(page):
    await grdm.open_edit_wiki(page)
    await grdm.fill_text(page, text)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['strongWiki', 'italicWiki', 'format_strikethrough'])
    
    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator("em > strong > del", has_text=text)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 新しく入力した「test」の文字を範囲指定して編集画面の太字ボタンと斜体ボタンと下線ボタンをそれぞれクリックし保存ボタンをクリック

入力した「test」が太文字、斜体、下線表示になること

In [ ]:
text = 'test 太文字、斜体、下線'
async def _step(page):
    await grdm.open_edit_wiki(page)
    await grdm.fill_text(page, text)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['strongWiki', 'italicWiki', 'format_underlined'])
    
    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator("em > strong > u", has_text=text)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 新しく入力した「test」の文字を範囲指定して編集画面の太字ボタンと斜体ボタンとフォントカラーボタンをそれぞれクリックし表示されているRGB設定画面のRの項目を255にして保存ボタンをクリック	

入力した「test」が太文字、斜体、フォントカラーが赤で表示になること

In [ ]:
text = 'test 太文字、斜体、フォントカラーが赤'
async def _step(page):
    await grdm.open_edit_wiki(page)
    await grdm.fill_text(page, text)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['strongWiki', 'italicWiki', 'format_color_text'])
    
    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator("em > strong > span", has_text=text)).to_have_css("color", "rgb(255, 0, 0)")

await run_pw(_step)

## 新しく入力した「test」の文字を範囲指定して編集画面の太字ボタンと取消線ボタンと下線ボタンをそれぞれクリックし保存ボタンをクリック

入力した「test」が太文字、取消、下線表示になること

In [ ]:
text = 'test 太文字、取消、下線'
async def _step(page):
    await grdm.open_edit_wiki(page)
    await grdm.fill_text(page, text)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['strongWiki', 'format_strikethrough', 'format_underlined'])
    
    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator("strong > u > del", has_text=text)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 新しく入力した「test」の文字を範囲指定して編集画面の太字ボタンと取消線ボタンとフォントカラーボタンをそれぞれクリックし表示されているRGB設定画面のRの項目を255にして保存ボタンをクリック	

入力した「test」が太文字、取消、フォントカラーが赤で表示になること

In [ ]:
text = 'test 太文字、取消、フォントカラーが赤'
async def _step(page):
    await grdm.open_edit_wiki(page)
    await grdm.fill_text(page, text)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['strongWiki', 'format_strikethrough', 'format_color_text'])
    
    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator("strong > span > del", has_text=text)).to_have_css("color", "rgb(255, 0, 0)")

await run_pw(_step)

## 新しく入力した「test」の文字を範囲指定して編集画面の太字ボタンと下線ボタンとフォントカラーボタンをそれぞれクリックし表示されているRGB設定画面のRの項目を255にして保存ボタンをクリック	

入力した「test」が太文字、下線、フォントカラーが赤で表示になること

In [ ]:
text = 'test 太文字、下線、フォントカラーが赤'
async def _step(page):
    await grdm.open_edit_wiki(page)
    await grdm.fill_text(page, text)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['strongWiki', 'format_underlined', 'format_color_text'])
    
    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator("strong > u > span", has_text=text)).to_have_css("color", "rgb(255, 0, 0)")

await run_pw(_step)

## 新しく入力した「test」の文字を範囲指定して編集画面の太字ボタンと斜体ボタンと取消線ボタンと下線ボタンをそれぞれクリックし保存ボタンをクリック

入力した「test」が太文字、斜体、取消、下線表示になること

In [ ]:
text = 'test 太文字、斜体、取消、下線'
async def _step(page):
    await grdm.open_edit_wiki(page)
    await grdm.fill_text(page, text)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['strongWiki', 'italicWiki', 'format_strikethrough', 'format_underlined'])
    
    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator("em> strong > u > del", has_text=text)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 新しく入力した「test」の文字を範囲指定して編集画面の太字ボタンと斜体ボタンと取消線ボタンとフォントカラーボタンをそれぞれクリックし表示されているRGB設定画面のRの項目を255にして保存ボタンをクリック

入力した「test」が太文字、斜体、取消、フォントカラーが赤で表示になること

In [ ]:
text = 'test 太文字、斜体、取消、フォントカラーが赤'
async def _step(page):
    await grdm.open_edit_wiki(page)
    await grdm.fill_text(page, text)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['strongWiki', 'italicWiki', 'format_strikethrough', 'format_color_text'])
    
    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator("em > strong > span > del", has_text=text)).to_have_css("color", "rgb(255, 0, 0)")

await run_pw(_step)

## 新しく入力した「test」の文字を範囲指定して編集画面の太字ボタンと斜体ボタンと下線ボタンとフォントカラーボタンをそれぞれクリックし表示されているRGB設定画面のRの項目を255にして保存ボタンをクリック

入力した「test」が太文字、斜体、下線、フォントカラーが赤で表示になること

In [ ]:
text = 'test 太文字、斜体、下線、フォントカラーが赤'
async def _step(page):
    await grdm.open_edit_wiki(page)
    await grdm.fill_text(page, text)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['strongWiki', 'italicWiki', 'format_underlined', 'format_color_text'])
    
    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator("em > strong > u > span", has_text=text)).to_have_css("color", "rgb(255, 0, 0)")

await run_pw(_step)

## 新しく入力した「test」の文字を範囲指定して編集画面の太字ボタンと取消線ボタンと下線ボタンとフォントカラーボタンをそれぞれクリックし表示されているRGB設定画面のRの項目を255にして保存ボタンをクリック

入力した「test」が太文字、取消線、下線、フォントカラーが赤で表示になること

In [ ]:
text = 'test 太文字、取消線、下線、フォントカラーが赤'
async def _step(page):
    await grdm.open_edit_wiki(page)
    await grdm.fill_text(page, text)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['strongWiki', 'format_strikethrough', 'format_underlined', 'format_color_text'])
    
    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator("strong > u > span > del", has_text=text)).to_have_css("color", "rgb(255, 0, 0)")

await run_pw(_step)

## 入力した「test」の文字を範囲指定して編集画面の太字ボタンと斜体ボタンと取消線ボタンと下線ボタンとフォントカラーボタンをそれぞれクリックし表示されているRGB設定画面のRの項目を255にして保存ボタンをクリック

入力した「test」が太文字、斜体、取消線、下線、フォントカラーが赤で表示になること

In [ ]:
text = 'test 太文字、斜体、取消線、下線、フォントカラーが赤'
async def _step(page):
    await grdm.open_edit_wiki(page)
    await grdm.fill_text(page, text)
    await grdm.select_text_range(page, text)
    await grdm.click_wiki_menu_save(page, ['strongWiki', 'italicWiki', 'format_strikethrough', 'format_underlined', 'format_color_text'])
    
    view_locator = page.locator('#mView .ProseMirror[contenteditable="false"]')
    await expect(view_locator).to_be_visible(timeout=transition_timeout)
    await expect(view_locator.locator("em > strong > u > span > del", has_text=text)).to_have_css("color", "rgb(255, 0, 0)")

await run_pw(_step)

## (指定がある場合) プロジェクトを削除する

In [ ]:
async def _step(page):
    if not delete_project:
        return
    await scripts.grdm.delete_project(page)
    await scripts.grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)

終了処理を実施。

In [ ]:
await finish_pw_context()

In [ ]:
!rm -fr {work_dir}